In [16]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import QueryRequest, Filter, FieldCondition, MatchValue

BACKEND_DIR = Path("..").resolve()
DATASET_DIR = BACKEND_DIR / "dataset"
KB_DIR = BACKEND_DIR / "knowledge_base"
REZULTATI_DIR = BACKEND_DIR / "rezultati"

QDRANT_URL = "http://localhost:6333"
KOLEKCIJA_SLIKE = "plant_images"
KOLEKCIJA_TEKST = "plant_text"
QDRANT_BATCH = 16

K = 200
PRAG_ABSTAIN_SLIKA = 0.59
TAU_SLIKA = 0.06

MODEL_TEKST = "BAAI/bge-large-en-v1.5"
PREFIKS_UPITA = "Represent this sentence for searching relevant passages: "
TEZINA_DOKUMENTA = 0.3
PRAG_ABSTAIN_TEKST = 0.6257
TAU_TEKST = 0.1162
BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DUZINA_LISTE = 3
CILJ_TACNOSTI = 95.0
MIN_VRSTA_U_CELIJI = 10

SEED = 42
SLIKA_PO_UPITU = 20
K_RRF_KANDIDATI = [1, 5, 10, 30, 60]
W_SLIKA_KANDIDATI = [0.5, 0.6, 0.7, 0.8]
K_RRF_OSNOVNI = 60
W_SLIKA_OSNOVNI = 0.5

client = QdrantClient(url=QDRANT_URL, timeout=60)

print("DEVICE =", DEVICE)
print("Tačaka u", KOLEKCIJA_SLIKE, ":", client.count(KOLEKCIJA_SLIKE).count)
print("Tačaka u", KOLEKCIJA_TEKST, ":", client.count(KOLEKCIJA_TEKST).count)

DEVICE = cuda
Tačaka u plant_images : 135877
Tačaka u plant_text : 2858


In [17]:
mapiranje = pd.read_csv(DATASET_DIR / "specie_id - scientific_name mapping.csv", dtype=str)
ime_vrste = dict(zip(mapiranje["specie_id"], mapiranje["scientific_name"]))
BROJ_VRSTA = len(ime_vrste)

val_slike = pd.read_csv(BACKEND_DIR / "eval_upiti.csv",
                        usecols=["image_path", "specie_id", "top1_specie_id"],
                        dtype={"specie_id": str, "top1_specie_id": str})

with np.load(BACKEND_DIR / "eval_val_vektori.npz") as sacuvano:
    vektor_slike = dict(zip(sacuvano["image_path"], sacuvano["vector"]))

vektori_slika = np.stack([vektor_slike[putanja] for putanja in val_slike["image_path"]])

with open(KB_DIR / "tekst_upiti.json", encoding="utf-8") as f:
    svi_upiti = json.load(f)

def ucitaj_upite(naziv):
    redovi = []
    for sid in svi_upiti:
        if sid == "_meta":
            continue
        for stavka in svi_upiti[sid][naziv]:
            redovi.append({"specie_id": sid, "stil": stavka["stil"], "upit": stavka["upit"]})
    return pd.DataFrame(redovi)

val_tekst = ucitaj_upite("val")

print("Vrsta:", BROJ_VRSTA)
print("Val slika:", len(val_slike), "| vektori:", vektori_slika.shape)
print("Val tekst upita:", len(val_tekst))



Vrsta: 324
Val slika: 29000 | vektori: (29000, 512)
Val tekst upita: 648


In [18]:
def slikovna_grana(vektori):
    rezultati = []
    for start in tqdm(range(0, len(vektori), QDRANT_BATCH), desc="Slikovna grana"):
        requests = []
        for vektor in vektori[start : start + QDRANT_BATCH]:
            requests.append(QueryRequest(query=vektor.tolist(), limit=K, with_payload=["specie_id"]))
        responses = client.query_batch_points(collection_name=KOLEKCIJA_SLIKE, requests=requests)

        for response in responses:
            species_scores = {}
            for point in response.points:
                species_scores.setdefault(point.payload["specie_id"], point.score)
            rang = list(species_scores)
            top1_score = species_scores[rang[0]]

            if len(rang) >= 2:
                gap = top1_score - species_scores[rang[1]]
            else:
                gap = top1_score - response.points[-1].score

            if top1_score < PRAG_ABSTAIN_SLIKA:
                ishod = "abstain"
            elif gap >= TAU_SLIKA:
                ishod = "siguran"
            else:
                ishod = "lista"

            rezultati.append({"rang": rang, "top1_score": top1_score, "gap": gap, "ishod": ishod})
    return rezultati


In [19]:
slika_rez = slikovna_grana(vektori_slika)

val_slike["ishod"] = [rezultat["ishod"] for rezultat in slika_rez]

udeli_slika = val_slike["ishod"].value_counts(normalize=True) * 100
print("Udeli odluka (%):", udeli_slika.round(2).to_dict())

Slikovna grana: 100%|██████████| 1813/1813 [03:14<00:00,  9.31it/s]

Udeli odluka (%): {'lista': 60.09, 'siguran': 37.71, 'abstain': 2.2}


In [20]:
model = SentenceTransformer(MODEL_TEKST, device=DEVICE)

srednji_vektor_val_upita = np.load(KB_DIR / "bge_srednji_vektori.npz")["srednji_vektor_val_upita"]


def centriraj(vektor, srednji_vektor):
    vektor = vektor - srednji_vektor
    return vektor / np.linalg.norm(vektor, axis=-1, keepdims=True)


def skor_po_vrsti(vektor_upita, nivo):
    odgovor = client.query_points_groups(
        collection_name=KOLEKCIJA_TEKST,
        query=vektor_upita.tolist(),
        using="centriran",
        query_filter=Filter(must=[FieldCondition(key="nivo", match=MatchValue(value=nivo))]),
        group_by="specie_id",
        limit=BROJ_VRSTA,
        group_size=1,
        with_payload=False,
    )
    skorovi = {}
    for grupa in odgovor.groups:
        skorovi[grupa.id] = grupa.hits[0].score
    return skorovi


def sirov_skor(sirovi_vektor_upita):
    odgovor = client.query_points(
        collection_name=KOLEKCIJA_TEKST,
        query=sirovi_vektor_upita.tolist(),
        using="sirov",
        query_filter=Filter(must=[FieldCondition(key="nivo", match=MatchValue(value="prozor"))]),
        limit=1,
        with_payload=False,
    )
    return odgovor.points[0].score


def tekstualna_grana(upiti):
    sirovi_vektori = model.encode([PREFIKS_UPITA + upit for upit in upiti],
                                  batch_size=BATCH_SIZE, normalize_embeddings=True)
    centrirani_vektori = centriraj(sirovi_vektori, srednji_vektor_val_upita)

    rezultati = []
    for i in tqdm(range(len(upiti)), desc="Tekstualna grana"):
        prozor = skor_po_vrsti(centrirani_vektori[i], "prozor")
        dokument = skor_po_vrsti(centrirani_vektori[i], "dokument")

        vrste = []
        for sid in prozor:
            vrste.append((sid, (1 - TEZINA_DOKUMENTA) * prozor[sid] + TEZINA_DOKUMENTA * dokument[sid]))
        vrste.sort(key=lambda par: par[1], reverse=True)

        sirov = sirov_skor(sirovi_vektori[i])
        gap = vrste[0][1] - vrste[1][1]

        if sirov < PRAG_ABSTAIN_TEKST:
            ishod = "abstain"
        elif gap >= TAU_TEKST:
            ishod = "siguran"
        else:
            ishod = "lista"

        rang = [sid for sid, _ in vrste]
        rezultati.append({"rang": rang, "sirov_score": sirov, "gap": gap, "ishod": ishod})
    return rezultati

Loading weights: 100%|██████████| 391/391 [00:04<00:00, 94.21it/s]


In [21]:
tekst_rez = tekstualna_grana(val_tekst["upit"].tolist())

val_tekst["ishod"] = [rezultat["ishod"] for rezultat in tekst_rez]
val_tekst["top1"] = [rezultat["rang"][0] for rezultat in tekst_rez]

hit1_tekst = (val_tekst["top1"] == val_tekst["specie_id"]).mean() * 100
udeli_tekst = val_tekst["ishod"].value_counts(normalize=True) * 100

print(f"Hit@1: {hit1_tekst:.2f}%")
print("Udeli odluka (%):", udeli_tekst.round(2).to_dict())


Tekstualna grana: 100%|██████████| 648/648 [00:59<00:00, 10.89it/s]

Hit@1: 55.25%
Udeli odluka (%): {'lista': 76.85, 'siguran': 23.15}


In [22]:
val_tekst_df=pd.DataFrame(val_tekst)

val_slike_df=pd.DataFrame(val_slike)


In [23]:
slike_po_vrsti = {}
for i, sid in enumerate(val_slike["specie_id"]):
    slike_po_vrsti.setdefault(sid, []).append(i)


def napravi_parove(tekst_vrste, slike_po_vrsti, broj_slika, seed):
    """tekst upit se upari sa najviše broj_slika različitih slika iste vrste"""
    rng = np.random.default_rng(seed)
    redovi = []
    for t, sid in enumerate(tekst_vrste):
        broj = min(broj_slika, len(slike_po_vrsti[sid]))
        izabrane_slike = rng.choice(slike_po_vrsti[sid], size=broj, replace=False)
        for s in izabrane_slike:
            redovi.append({"tekst_i": t, "slika_i": int(s), "specie_id": sid})
    return pd.DataFrame(redovi)


def pozicija(vrsta, rang):
    """mesto vrste u listi: '1', '2', '3' ili 'nema'"""
    if vrsta in rang[:DUZINA_LISTE]:
        return str(rang.index(vrsta) + 1)
    return "nema"


def dodaj_kolone_grana(parovi, slika_rezultati, tekst_rezultati):
    ishod_slika, ishod_tekst, top1_slika, top1_tekst = [], [], [], []
    poz_slike_u_tekstu, poz_teksta_u_slici = [], []
    for slika_i, tekst_i in zip(parovi["slika_i"], parovi["tekst_i"]):
        s = slika_rezultati[slika_i]
        t = tekst_rezultati[tekst_i]
        ishod_slika.append(s["ishod"])
        ishod_tekst.append(t["ishod"])
        top1_slika.append(s["rang"][0])
        top1_tekst.append(t["rang"][0])
        poz_slike_u_tekstu.append(pozicija(s["rang"][0], t["rang"]))
        poz_teksta_u_slici.append(pozicija(t["rang"][0], s["rang"]))

    parovi = parovi.copy()
    parovi["ishod_slika"] = ishod_slika
    parovi["ishod_tekst"] = ishod_tekst
    parovi["top1_slika"] = top1_slika
    parovi["top1_tekst"] = top1_tekst
    parovi["poz_slike_u_tekstu"] = poz_slike_u_tekstu
    parovi["poz_teksta_u_slici"] = poz_teksta_u_slici
    parovi["slika_tacna"] = parovi["top1_slika"] == parovi["specie_id"]
    parovi["tekst_tacan"] = parovi["top1_tekst"] == parovi["specie_id"]
    return parovi


parovi = napravi_parove(val_tekst["specie_id"], slike_po_vrsti, SLIKA_PO_UPITU, SEED)

parovi["stil"] = val_tekst["stil"].to_numpy()[parovi["tekst_i"]]
parovi = dodaj_kolone_grana(parovi, slika_rez, tekst_rez)

assert not parovi.duplicated(["tekst_i", "slika_i"]).any(), "par (upit, slika) se ponavlja"
print("Parova po vrsti: od", parovi.groupby("specie_id").size().min(), "do", parovi.groupby("specie_id").size().max())
print("Parova:", len(parovi), "| tekst upita:", parovi["tekst_i"].nunique(),
      "| vrsta:", parovi["specie_id"].nunique(), "| različitih slika:", parovi["slika_i"].nunique())
parovi.head(5)

Parova po vrsti: od 18 do 40
Parova: 11590 | tekst upita: 648 | vrsta: 324 | različitih slika: 8442


,tekst_i,slika_i,specie_id,stil,ishod_slika,ishod_tekst,top1_slika,top1_tekst,poz_slike_u_tekstu,poz_teksta_u_slici,slika_tacna,tekst_tacan
0,0,14001,0000,short,siguran,lista,0247,0139,3,nema,False,False
1,0,17001,0000,short,lista,lista,0066,0139,nema,nema,False,False
2,0,3001,0000,short,lista,lista,0247,0139,3,nema,False,False
3,0,22004,0000,short,lista,lista,0040,0139,nema,nema,False,False
4,0,27000,0000,short,lista,lista,0000,0139,nema,nema,True,False


In [24]:
def prve_tri(rang):
    imena = []
    for sid in rang[:3]:
        imena.append(ime_vrste[sid])
    return ", ".join(imena)


pregled = pd.DataFrame({
    "tekst_i": parovi["tekst_i"],
    "slika_i": parovi["slika_i"],
    "vrsta": [ime_vrste[sid] for sid in parovi["specie_id"]],
    "stil": parovi["stil"],
    "upit": val_tekst["upit"].to_numpy()[parovi["tekst_i"]],
    "slika": val_slike["image_path"].to_numpy()[parovi["slika_i"]],
    "ishod_slika": parovi["ishod_slika"],
    "top3_slika": [prve_tri(slika_rez[i]["rang"]) for i in parovi["slika_i"]],
    "slika_tacna": parovi["slika_tacna"],
    "ishod_tekst": parovi["ishod_tekst"],
    "top3_tekst": [prve_tri(tekst_rez[i]["rang"]) for i in parovi["tekst_i"]],
    "tekst_tacan": parovi["tekst_tacan"],
    "poz_slike_u_tekstu": parovi["poz_slike_u_tekstu"],
    "poz_teksta_u_slici": parovi["poz_teksta_u_slici"],
})

PAROVI_CSV = REZULTATI_DIR / "parovi_upita.csv"
pregled.to_csv(PAROVI_CSV, index=False, encoding="utf-8-sig")
print("Snimljeno", len(pregled), "parova u", PAROVI_CSV)

Snimljeno 11590 parova u C:\Users\mika\Desktop\RAG\projekat\backend\rezultati\parovi_upita.csv


In [25]:
ISHODI = ["abstain", "siguran", "lista"]

tabela_3x3 = pd.crosstab(parovi["ishod_slika"], parovi["ishod_tekst"],
                         normalize=True, margins=True, margins_name="ukupno") * 100
tabela_3x3 = tabela_3x3.reindex(index=ISHODI + ["ukupno"], columns=ISHODI + ["ukupno"], fill_value=0.0)

tabela_3x3.index = ["slika: abstain", "slika: siguran", "slika: lista", "ukupno"]
tabela_3x3.columns = ["tekst: abstain", "tekst: siguran", "tekst: lista", "ukupno"]
tabela_3x3.round(1)

,tekst: abstain,tekst: siguran,tekst: lista,ukupno
slika: abstain,0.0,0.6,2.2,2.8
slika: siguran,0.0,10.0,29.8,39.8
slika: lista,0.0,12.2,45.3,57.4
ukupno,0.0,22.8,77.2,100.0


In [26]:
celije = {}


def dodaj_celiju(kljuc, slucaj, opis, podskup, kolona_tacnosti):
    if len(podskup) > 0:
        tacnost = podskup[kolona_tacnosti].mean() * 100
    else:
        tacnost = np.nan
        
    celije[kljuc] = {
        "slucaj": slucaj,
        "opis": opis,
        "parova": len(podskup),
        "udeo_svih_%": len(podskup) / len(parovi) * 100,
        "tekst_upita": podskup["tekst_i"].nunique(),
        "vrsta": podskup["specie_id"].nunique(),
        "tacnost_%": tacnost,
    }

In [27]:
obe_sigurne = parovi[(parovi["ishod_slika"] == "siguran") & (parovi["ishod_tekst"] == "siguran")]

slazu_se = obe_sigurne[obe_sigurne["top1_slika"] == obe_sigurne["top1_tekst"]]

ne_slazu_se = obe_sigurne[obe_sigurne["top1_slika"] != obe_sigurne["top1_tekst"]]

dodaj_celiju("obe_sigurne_slazu_se", 4, "obe sigurne, ista vrsta", slazu_se, "slika_tacna")


print(f"Obe grane sigurne: {len(obe_sigurne)} parova ({len(obe_sigurne) / len(parovi):.1%} svih)")
print(f"  slažu se:    {len(slazu_se)} parova, vrsta tačna u {slazu_se['slika_tacna'].mean():.1%}")


print(f"  ne slažu se: {len(ne_slazu_se)} parova ({len(ne_slazu_se) / len(obe_sigurne):.1%} od obe sigurne), "
      f"iz {ne_slazu_se['tekst_i'].nunique()} tekst upita i {ne_slazu_se['specie_id'].nunique()} vrsta\n")


slika_po_parovima = ne_slazu_se["slika_tacna"].mean() * 100
tekst_po_parovima = ne_slazu_se["tekst_tacan"].mean() * 100

print("Kad se ne slažu:")
print(f"  slika tačna:  {slika_po_parovima:.1f}%")
print(f"  tekst tačan:  {tekst_po_parovima:.1f}%")
print(f"  obe pogrešne: {(~ne_slazu_se['slika_tacna'] & ~ne_slazu_se['tekst_tacan']).mean():.1%}")

if slika_po_parovima > tekst_po_parovima:
    PREDNOST_KAD_SE_NE_SLAZU = "slika"
elif tekst_po_parovima > slika_po_parovima:
    PREDNOST_KAD_SE_NE_SLAZU = "tekst"
else:
    PREDNOST_KAD_SE_NE_SLAZU = "nijedna"
print("\nPREDNOST_KAD_SE_NE_SLAZU =", PREDNOST_KAD_SE_NE_SLAZU)

Obe grane sigurne: 1158 parova (10.0% svih)
  slažu se:    1054 parova, vrsta tačna u 100.0%
  ne slažu se: 104 parova (9.0% od obe sigurne), iz 30 tekst upita i 22 vrsta

Kad se ne slažu:
  slika tačna:  66.3%
  tekst tačan:  31.7%
  obe pogrešne: 1.9%

PREDNOST_KAD_SE_NE_SLAZU = slika


In [28]:

POZICIJE = ["1", "2", "3", "nema"]

slika_sig_tekst_lista = parovi[(parovi["ishod_slika"] == "siguran") & (parovi["ishod_tekst"] == "lista")]

for poz in POZICIJE:
    podskup = slika_sig_tekst_lista[slika_sig_tekst_lista["poz_slike_u_tekstu"] == poz]
    dodaj_celiju(f"slika_sigurna_{poz}", 5, f"slika sigurna, u listi teksta: {poz}", podskup, "slika_tacna")

tekst_sig_slika_lista = parovi[(parovi["ishod_tekst"] == "siguran") & (parovi["ishod_slika"] == "lista")]

for poz in POZICIJE:
    podskup = tekst_sig_slika_lista[tekst_sig_slika_lista["poz_teksta_u_slici"] == poz]
    dodaj_celiju(f"tekst_siguran_{poz}", 5, f"tekst siguran, u listi slike: {poz}", podskup, "tekst_tacan")

print(f"Slika sigurna + tekst lista: {len(slika_sig_tekst_lista) / len(parovi):.1%} parova, "
      f"vrsta slike tačna ukupno {slika_sig_tekst_lista['slika_tacna'].mean():.1%}")

print(f"Tekst siguran + slika lista: {len(tekst_sig_slika_lista) / len(parovi):.1%} parova, "
      f"vrsta teksta tačna ukupno {tekst_sig_slika_lista['tekst_tacan'].mean():.1%}\n")

kolone = ["opis", "parova", "vrsta", "tacnost_%"]
tabela_5 = pd.DataFrame.from_dict(celije, orient="index")
tabela_5.loc[[kljuc for kljuc in celije if kljuc.startswith(("slika_sig", "tekst_sig"))], kolone].round(1)

Slika sigurna + tekst lista: 29.8% parova, vrsta slike tačna ukupno 95.7%
Tekst siguran + slika lista: 12.2% parova, vrsta teksta tačna ukupno 96.0%



,opis,parova,vrsta,tacnost_%
slika_sigurna_1,"slika sigurna, u listi teksta: 1",1567,155,99.8
slika_sigurna_2,"slika sigurna, u listi teksta: 2",409,60,99.5
slika_sigurna_3,"slika sigurna, u listi teksta: 3",215,35,95.8
slika_sigurna_nema,"slika sigurna, u listi teksta: nema",1260,162,89.2
tekst_siguran_1,"tekst siguran, u listi slike: 1",793,108,100.0
tekst_siguran_2,"tekst siguran, u listi slike: 2",377,79,99.5
tekst_siguran_3,"tekst siguran, u listi slike: 3",84,41,97.6
tekst_siguran_nema,"tekst siguran, u listi slike: nema",156,53,66.7


In [29]:
obe_liste = parovi[(parovi["ishod_slika"] == "lista") & (parovi["ishod_tekst"] == "lista")]

isti_top1 = obe_liste[obe_liste["top1_slika"] == obe_liste["top1_tekst"]]

print(f"Obe liste: {len(obe_liste) / len(parovi):.1%} parova")
print(f"  ista prva vrsta u {len(isti_top1) / len(obe_liste):.1%} od njih, tada tačna u {isti_top1['slika_tacna'].mean():.1%}")
print(f"  (za poređenje: prva vrsta slike tačna u {obe_liste['slika_tacna'].mean():.1%}, "
      f"teksta u {obe_liste['tekst_tacan'].mean():.1%} svih parova sa dve liste)")

Obe liste: 45.3% parova
  ista prva vrsta u 27.4% od njih, tada tačna u 92.6%
  (za poređenje: prva vrsta slike tačna u 62.2%, teksta u 39.0% svih parova sa dve liste)


In [30]:
celije_df = pd.DataFrame.from_dict(celije, orient="index")
celije_df.index.name = "kljuc"

celije_df["siguran"] = (celije_df["tacnost_%"] >= CILJ_TACNOSTI) & (celije_df["vrsta"] >= MIN_VRSTA_U_CELIJI)
PRAVILA_SIGURAN = celije_df["siguran"].to_dict()

celije_df.to_csv(REZULTATI_DIR / "fuzija_celije.csv", encoding="utf-8-sig")
celije_df.round(1)

,slucaj,opis,parova,udeo_svih_%,tekst_upita,vrsta,tacnost_%,siguran
kljuc,,,,,,,,
obe_sigurne_slazu_se,4,"obe sigurne, ista vrsta",1054,9.1,127,98,100.0,True
slika_sigurna_1,5,"slika sigurna, u listi teksta: 1",1567,13.5,188,155,99.8,True
slika_sigurna_2,5,"slika sigurna, u listi teksta: 2",409,3.5,65,60,99.5,True
slika_sigurna_3,5,"slika sigurna, u listi teksta: 3",215,1.9,37,35,95.8,True
slika_sigurna_nema,5,"slika sigurna, u listi teksta: nema",1260,10.9,216,162,89.2,False
tekst_siguran_1,5,"tekst siguran, u listi slike: 1",793,6.8,136,108,100.0,True
tekst_siguran_2,5,"tekst siguran, u listi slike: 2",377,3.3,99,79,99.5,True
tekst_siguran_3,5,"tekst siguran, u listi slike: 3",84,0.7,49,41,97.6,True
tekst_siguran_nema,5,"tekst siguran, u listi slike: nema",156,1.3,64,53,66.7,False


In [32]:
def rrf(rang_slika, rang_tekst, k_rrf, w_slika):
    poeni = {}
    for mesto, sid in enumerate(rang_slika, start=1):
        novi_poeni= w_slika / (k_rrf + mesto)
        poeni[sid] = poeni.get(sid, 0.0) + novi_poeni
    for mesto, sid in enumerate(rang_tekst, start=1):
        novi_poeni = (1 - w_slika) / (k_rrf + mesto)
        poeni[sid] = poeni.get(sid, 0.0) + novi_poeni
    return sorted(poeni, key=poeni.get, reverse=True)


PORUKA_NEMA_ODGOVORA = "Neuspela identifikacija biljke. Pokušajte ponovo sa boljom slikom i/ili tekstom."

def fuzija(slika, tekst, k_rrf=K_RRF_OSNOVNI, w_slika=W_SLIKA_OSNOVNI):
    """slika, tekst: rezultat grane ili None ako taj deo upita nije poslat"""
    ishod_s = slika["ishod"] if slika is not None else "nije_poslato"
    ishod_t = tekst["ishod"] if tekst is not None else "nije_poslato"

    bez_slike = ishod_s in ("abstain", "nije_poslato")
    bez_teksta = ishod_t in ("abstain", "nije_poslato")

    if bez_slike and bez_teksta:
        return {"ishod": "nema_odgovora", "prikaz": [], "izvor": "nijedna", "obrazlozenje": PORUKA_NEMA_ODGOVORA}

    if bez_slike or bez_teksta:
        if bez_teksta:
            grana, rezultat, obrazlozenje = "slika", slika, "samo na osnovu slike"
        else:
            grana, rezultat, obrazlozenje = "tekst", tekst, "samo na osnovu opisa"
        if rezultat["ishod"] == "siguran":
            return {"ishod": "siguran", "prikaz": rezultat["rang"][:1], "izvor": grana, "obrazlozenje": obrazlozenje}
        return {"ishod": "lista", "prikaz": rezultat["rang"][:DUZINA_LISTE], "izvor": grana, "obrazlozenje": obrazlozenje}

    top1_s = slika["rang"][0] 
    top1_t = tekst["rang"][0]

    if ishod_s == "siguran" and ishod_t == "siguran" and top1_s != top1_t:
        if PREDNOST_KAD_SE_NE_SLAZU == "tekst":
            prikaz = [top1_t, top1_s]
            obrazlozenje = "grane se ne slažu; verovatnija je vrsta iz opisa"
        elif PREDNOST_KAD_SE_NE_SLAZU == "slika":
            prikaz = [top1_s, top1_t]
            obrazlozenje = "grane se ne slažu; verovatnija je vrsta sa slike"
        else:
            prikaz = [top1_s, top1_t]
            obrazlozenje = "grane se ne slažu; obe vrste su podjednako moguće"
        return {"ishod": "lista", "prikaz": prikaz, "izvor": "neslaganje", "obrazlozenje": obrazlozenje}

    if ishod_s == "siguran" and ishod_t == "siguran":
        if PRAVILA_SIGURAN["obe_sigurne_slazu_se"]:
            return {"ishod": "siguran", "prikaz": [top1_s], "izvor": "obe",
                    "obrazlozenje": "slika i opis daju istu vrstu"}

    if ishod_s == "siguran" and ishod_t == "lista":
        mesto = pozicija(top1_s, tekst["rang"])
        if PRAVILA_SIGURAN["slika_sigurna_" + mesto]:
            return {"ishod": "siguran", "prikaz": [top1_s], "izvor": "obe",
                    "obrazlozenje": "slika je sigurna, a ista vrsta je i među predlozima po opisu"}

    if ishod_t == "siguran" and ishod_s == "lista":
        mesto = pozicija(top1_t, slika["rang"])
        if PRAVILA_SIGURAN["tekst_siguran_" + mesto]:
            return {"ishod": "siguran", "prikaz": [top1_t], "izvor": "obe",
                    "obrazlozenje": "opis je siguran, a ista vrsta je i među predlozima po slici"}

    lista = rrf(slika["rang"], tekst["rang"], k_rrf, w_slika)[:DUZINA_LISTE]
    return {"ishod": "lista", "prikaz": lista, "izvor": "rrf",
            "obrazlozenje": "grane se delimično slažu; lista je spojena iz obe grane"}

In [ ]:
def imena(vrste):
    return ", ".join(ime_vrste[sid] for sid in vrste)

primeri = parovi.drop_duplicates(["ishod_slika", "ishod_tekst"]).sort_values(["ishod_slika", "ishod_tekst"])
for _, par in primeri.iterrows():
    s = slika_rez[par["slika_i"]]
    t = tekst_rez[par["tekst_i"]]
    odgovor = fuzija(s, t)
    print(f"slika: {par['ishod_slika']:8s} | tekst: {par['ishod_tekst']:8s} | tačna vrsta: {ime_vrste[par['specie_id']]}")
    print(f"   slika top3: {imena(s['rang'][:3])}")
    print(f"   tekst top3: {imena(t['rang'][:3])}")
    print(f"   FUZIJA -> {odgovor['ishod']}: {imena(odgovor['prikaz'])}   ({odgovor['obrazlozenje']})\n")

slika: abstain  | tekst: lista    | tačna vrsta: Lactuca virosa
   slika top3: Valeriana rubra, Cirsium arvense, Helminthotheca echioides
   tekst top3: Hyoscyamus niger, Aralia nudicaulis, Lactuca serriola
   FUZIJA -> lista: Hyoscyamus niger, Aralia nudicaulis, Lactuca serriola   (samo na osnovu opisa)

slika: abstain  | tekst: siguran  | tačna vrsta: Lavandula dentata
   slika top3: Lavandula dentata, Acacia dealbata, Acacia mearnsii
   tekst top3: Lavandula dentata, Salvia yangii, Lavandula angustifolia
   FUZIJA -> siguran: Lavandula dentata   (samo na osnovu opisa)

slika: lista    | tekst: lista    | tačna vrsta: Lactuca virosa
   slika top3: Mercurialis perennis, Erechtites hieraciifolius, Lactuca virosa
   tekst top3: Hyoscyamus niger, Aralia nudicaulis, Lactuca serriola
   FUZIJA -> lista: Lactuca virosa, Cicerbita alpina, Cirsium arvense   (grane se delimično slažu; lista je spojena iz obe grane)

slika: lista    | tekst: siguran  | tačna vrsta: Phyllanthus tenellus
   slika

In [ ]:
koristi_rrf = []
for slika_i, tekst_i in zip(parovi["slika_i"], parovi["tekst_i"]):
    koristi_rrf.append(fuzija(slika_rez[slika_i], tekst_rez[tekst_i])["izvor"] == "rrf")
rrf_parovi = parovi[koristi_rrf].reset_index(drop=True)
print(f"Parova sa RRF listom: {len(rrf_parovi)} ({len(rrf_parovi) / len(parovi):.1%} svih)\n")

mreza = []
pogoci = {}   
for k_rrf in K_RRF_KANDIDATI:
    for w_slika in W_SLIKA_KANDIDATI:
        hit1, hit3 = [], []
        for slika_i, tekst_i, sid in zip(rrf_parovi["slika_i"], rrf_parovi["tekst_i"], rrf_parovi["specie_id"]):
            lista = rrf(slika_rez[slika_i]["rang"], tekst_rez[tekst_i]["rang"], k_rrf, w_slika)[:DUZINA_LISTE]
            hit1.append(lista[0] == sid)
            hit3.append(sid in lista)
        pogoci[(k_rrf, w_slika)] = np.array(hit3)
        mreza.append({"k": k_rrf, "w_slika": w_slika, "Hit@1_%": np.mean(hit1) * 100, "Hit@3_%": np.mean(hit3) * 100})
mreza_df = pd.DataFrame(mreza)
mreza_df.to_csv(REZULTATI_DIR / "fuzija_rrf_mreza.csv", index=False, encoding="utf-8-sig")

hit3_slika = np.mean([sid in slika_rez[i]["rang"][:DUZINA_LISTE] for i, sid in zip(rrf_parovi["slika_i"], rrf_parovi["specie_id"])]) * 100
hit3_tekst = np.mean([sid in tekst_rez[i]["rang"][:DUZINA_LISTE] for i, sid in zip(rrf_parovi["tekst_i"], rrf_parovi["specie_id"])]) * 100
print(f"Reference na istim parovima: lista same slike Hit@3 = {hit3_slika:.1f}%, lista samog teksta Hit@3 = {hit3_tekst:.1f}%\n")

print("Hit@3 (%) — redovi k, kolone težina slike w:")
mreza_df.pivot(index="k", columns="w_slika", values="Hit@3_%").round(1)

Parova sa RRF listom: 6661 (57.5% svih)

Reference na istim parovima: lista same slike Hit@3 = 90.8%, lista samog teksta Hit@3 = 52.6%

Hit@3 (%) — redovi k, kolone težina slike w:


w_slika,0.5,0.6,0.7,0.8
k,,,,
1,94.4,94.8,95.6,93.8
5,96.8,96.8,95.8,94.3
10,97.1,97.1,96.4,95.0
30,95.2,95.9,96.4,96.4
60,94.5,95.0,95.5,96.3


In [37]:
najbolji = mreza_df.sort_values(["Hit@3_%", "Hit@1_%"], ascending=False).iloc[0]
K_RRF, W_SLIKA = int(najbolji["k"]), float(najbolji["w_slika"])

hit_najbolji = pogoci[(K_RRF, W_SLIKA)]
hit_osnovni = pogoci[(K_RRF_OSNOVNI, W_SLIKA_OSNOVNI)]
dobitak = (hit_najbolji & ~hit_osnovni).sum()
gubitak = (~hit_najbolji & hit_osnovni).sum()

print(f"Najbolja kombinacija: k = {K_RRF}, w = {W_SLIKA} -> Hit@3 = {hit_najbolji.mean() * 100:.2f}%")
print(f"Standardni RRF:       k = {K_RRF_OSNOVNI}, w = {W_SLIKA_OSNOVNI} -> Hit@3 = {hit_osnovni.mean() * 100:.2f}%")
print(f"Na istih {len(hit_najbolji)} parova: dobitak {dobitak} parova, gubitak {gubitak} parova")
print(f"\nK_RRF = {K_RRF}, W_SLIKA = {W_SLIKA}")

Najbolja kombinacija: k = 10, w = 0.6 -> Hit@3 = 97.10%
Standardni RRF:       k = 60, w = 0.5 -> Hit@3 = 94.49%
Na istih 6661 parova: dobitak 230 parova, gubitak 56 parova

K_RRF = 10, W_SLIKA = 0.6


In [38]:
def samo_jedna_grana(rezultat):
    if rezultat["ishod"] == "abstain":
        return {"ishod": "nema_odgovora", "prikaz": []}
    if rezultat["ishod"] == "siguran":
        return {"ishod": "siguran", "prikaz": rezultat["rang"][:1]}
    return {"ishod": "lista", "prikaz": rezultat["rang"][:DUZINA_LISTE]}


def dodaj_odgovore_sistema(parovi, slika_rezultati, tekst_rezultati):
    odgovori = {"slika": [], "tekst": [], "fuzija": []}
    for slika_i, tekst_i in zip(parovi["slika_i"], parovi["tekst_i"]):
        s = slika_rezultati[slika_i]
        t = tekst_rezultati[tekst_i]
        odgovori["slika"].append(samo_jedna_grana(s))
        odgovori["tekst"].append(samo_jedna_grana(t))
        odgovori["fuzija"].append(fuzija(s, t, K_RRF, W_SLIKA))

    parovi = parovi.copy()
    for sistem, lista_odgovora in odgovori.items():
        parovi[f"{sistem}_odgovor"] = [odgovor["ishod"] for odgovor in lista_odgovora]
        parovi[f"{sistem}_prikaz"] = ["|".join(odgovor["prikaz"]) for odgovor in lista_odgovora]
        parovi[f"{sistem}_duzina"] = [len(odgovor["prikaz"]) for odgovor in lista_odgovora]
        parovi[f"{sistem}_pogodak"] = [sid in odgovor["prikaz"] for sid, odgovor in zip(parovi["specie_id"], lista_odgovora)]
    parovi["fuzija_izvor"] = [odgovor["izvor"] for odgovor in odgovori["fuzija"]]
    return parovi


parovi = dodaj_odgovore_sistema(parovi, slika_rez, tekst_rez)

za_snimanje = parovi.drop(columns=["slika_prikaz", "tekst_prikaz", "fuzija_prikaz"]).copy()
za_snimanje["image_path"] = val_slike["image_path"].to_numpy()[parovi["slika_i"]]
za_snimanje["upit"] = val_tekst["upit"].to_numpy()[parovi["tekst_i"]]
za_snimanje["fuzija_prikaz"] = [" | ".join(ime_vrste[sid] for sid in prikaz.split("|") if sid) for prikaz in parovi["fuzija_prikaz"]]
za_snimanje.to_csv(REZULTATI_DIR / "fuzija_parovi.csv", index=False, encoding="utf-8-sig")

print("Odakle dolazi odgovor fuzije (%):")
print((parovi["fuzija_izvor"].value_counts(normalize=True) * 100).round(1).to_string())

Odakle dolazi odgovor fuzije (%):
fuzija_izvor
rrf           57.5
obe           38.8
tekst          2.8
neslaganje     0.9


In [40]:
SISTEMI = {"slika": "samo slika", "tekst": "samo tekst", "fuzija": "fuzija"}


def mere_sistema(df, sistem):
    odgovor = df[f"{sistem}_odgovor"]
    pogodak = df[f"{sistem}_pogodak"]
    siguran = odgovor == "siguran"
    lista = odgovor == "lista"
    odgovoreno = odgovor != "nema_odgovora"
    return {
        "odbijeno_%": (~odgovoreno).mean() * 100,
        "siguran_%": siguran.mean() * 100,
        "tacnost_sigurnih_%": pogodak[siguran].mean() * 100,
        "siguran_a_pogresan_%": (siguran & ~pogodak).mean() * 100,
        "lista_%": lista.mean() * 100,
        "tacna_u_listi_%": pogodak[lista].mean() * 100,
        "duzina_liste": df.loc[lista, f"{sistem}_duzina"].mean(),
        "uspeh_odgovorenih_%": pogodak[odgovoreno].mean() * 100,
        "uspeh_svih_%": pogodak.mean() * 100,
    }


kolone = {}
for sistem, naziv in SISTEMI.items():
    kolone[naziv] = mere_sistema(parovi, sistem)
sistem_df = pd.DataFrame(kolone)

for mera in ["siguran_a_pogresan_%", "uspeh_odgovorenih_%"]:
    sistem_df.loc[f"razlika fuzija − slika: {mera}", "fuzija"] = sistem_df.loc[mera, "fuzija"] - sistem_df.loc[mera, "samo slika"]

sistem_df.to_csv(REZULTATI_DIR / "fuzija_sistem.csv", encoding="utf-8-sig")
sistem_df.round(2)

,samo slika,samo tekst,fuzija
odbijeno_%,2.81,0.00,0.00
siguran_%,39.77,22.79,39.45
tacnost_sigurnih_%,95.99,95.15,99.58
siguran_a_pogresan_%,1.60,1.10,0.16
lista_%,57.42,77.21,60.55
tacna_u_listi_%,91.51,64.39,96.18
duzina_liste,2.93,3.00,2.99
uspeh_odgovorenih_%,93.34,71.40,97.52
uspeh_svih_%,90.72,71.40,97.52
razlika fuzija − slika: siguran_a_pogresan_%,NaN,NaN,-1.43


In [41]:
po_celijama = parovi.groupby(["ishod_slika", "ishod_tekst"]).agg(
    parova=("specie_id", "size"),
    fuzija_siguran_pct=("fuzija_odgovor", lambda odgovor: (odgovor == "siguran").mean() * 100),
    uspeh_slika_pct=("slika_pogodak", "mean"),
    uspeh_fuzija_pct=("fuzija_pogodak", "mean"),
)
po_celijama["uspeh_slika_pct"] *= 100
po_celijama["uspeh_fuzija_pct"] *= 100
print("Po ćelijama (uspeh = tačna vrsta među prikazanima, odbijeni = neuspeh):")
print(po_celijama.round(1).to_string(), "\n")

KLJUCNE_MERE = ["siguran_%", "tacnost_sigurnih_%", "siguran_a_pogresan_%", "tacna_u_listi_%", "uspeh_odgovorenih_%"]
po_stilu = {}
for stil in ["short", "detailed"]:
    podskup = parovi[parovi["stil"] == stil]
    for sistem, naziv in SISTEMI.items():
        po_stilu[(stil, naziv)] = pd.Series(mere_sistema(podskup, sistem))[KLJUCNE_MERE]
print("Po stilu upita:")
pd.DataFrame(po_stilu).round(1)

Po ćelijama (uspeh = tačna vrsta među prikazanima, odbijeni = neuspeh):
                         parova  fuzija_siguran_pct  uspeh_slika_pct  uspeh_fuzija_pct
ishod_slika ishod_tekst                                                               
abstain     lista           253                 0.0              0.0              71.1
            siguran          73               100.0              0.0              98.6
lista       lista          5245                 0.0             91.3              97.0
            siguran        1410                88.9             92.3              98.6
siguran     lista          3451                63.5             95.7              99.0
            siguran        1158                91.0             97.0              99.8 

Po stilu upita:


short                     detailed                  
                     samo slika samo tekst fuzija samo slika samo tekst fuzija
siguran_%                  40.2       16.1   30.9       39.3       29.5   48.0
tacnost_sigurnih_%         95.7       89.7   99.4       96.3       98.1   99.7
siguran_a_pogresan_%        1.7        1.7    0.2        1.4        0.6    0.1
tacna_u_listi_%            91.4       51.4   95.6       91.6       79.9   97.0
uspeh_odgovorenih_%        93.2       57.5   96.8       93.5       85.2   98.3

In [42]:
with open(KB_DIR / "tekst_upiti_ai.json", encoding="utf-8") as f:
    laicki = json.load(f)["upiti"]

laicki_df = pd.DataFrame({"specie_id": list(laicki), "upit": list(laicki.values())})
laicki_rez = tekstualna_grana(laicki_df["upit"].tolist())

laicki_parovi = napravi_parove(laicki_df["specie_id"], slike_po_vrsti, SLIKA_PO_UPITU, SEED)
laicki_parovi = dodaj_kolone_grana(laicki_parovi, slika_rez, laicki_rez)
laicki_parovi = dodaj_odgovore_sistema(laicki_parovi, slika_rez, laicki_rez)

hit1_laicki = np.mean([rezultat["rang"][0] == sid for rezultat, sid in zip(laicki_rez, laicki_df["specie_id"])]) * 100
print(f"Laičkih upita: {len(laicki_df)} | Hit@1 teksta: {hit1_laicki:.1f}% | "
      f"odluke teksta: {pd.Series([r['ishod'] for r in laicki_rez]).value_counts().to_dict()}\n")

pd.DataFrame({naziv: mere_sistema(laicki_parovi, sistem) for sistem, naziv in SISTEMI.items()}).round(1)

Tekstualna grana: 100%|██████████| 18/18 [00:04<00:00,  4.23it/s]


Laičkih upita: 18 | Hit@1 teksta: 33.3% | odluke teksta: {'lista': 15, 'siguran': 3}



,samo slika,samo tekst,fuzija
odbijeno_%,2.2,0.0,0.0
siguran_%,34.7,16.7,18.6
tacnost_sigurnih_%,96.0,33.3,95.5
siguran_a_pogresan_%,1.4,11.1,0.8
lista_%,63.1,83.3,81.4
tacna_u_listi_%,96.5,33.3,96.2
duzina_liste,2.9,3.0,2.9
uspeh_odgovorenih_%,96.3,33.3,96.1
uspeh_svih_%,94.2,33.3,96.1


In [43]:
with open(KB_DIR / "tekst_upiti_van_teme.json", encoding="utf-8") as f:
    van_teme = json.load(f)["upiti"]
van_teme_rez = tekstualna_grana(van_teme)
print("Odluke teksta za upite van teme:", pd.Series([r["ishod"] for r in van_teme_rez]).value_counts().to_dict())

rng_v = np.random.default_rng(SEED)
redovi = []
for t in range(len(van_teme)):
    for s in rng_v.choice(len(val_slike), size=SLIKA_PO_UPITU, replace=False):
        redovi.append({"tekst_i": t, "slika_i": int(s), "specie_id": val_slike.loc[s, "specie_id"]})
van_teme_parovi = dodaj_odgovore_sistema(pd.DataFrame(redovi), slika_rez, van_teme_rez)
isto = ((van_teme_parovi["fuzija_odgovor"] == van_teme_parovi["slika_odgovor"])
        & (van_teme_parovi["fuzija_prikaz"] == van_teme_parovi["slika_prikaz"])).mean() * 100
print(f"\n(a) van teme + biljka iz baze: {len(van_teme_parovi)} parova, fuzija ista kao sama slika u {isto:.1f}%")

nisu_biljke = pd.read_csv(BACKEND_DIR / "eval_nisu_biljke.csv")
id_po_imenu = {ime: sid for sid, ime in ime_vrste.items()}
nisu_biljke_rez = []
for _, red in nisu_biljke.iterrows():
    if red["top1_score"] < PRAG_ABSTAIN_SLIKA:
        ishod = "abstain"
    elif red["gap"] >= TAU_SLIKA:
        ishod = "siguran"
    else:
        ishod = "lista"
    nisu_biljke_rez.append({"rang": [id_po_imenu.get(red["top1_name"], "?")], "ishod": ishod})

odgovori_b = []
for t in range(len(van_teme)):
    for s in range(len(nisu_biljke_rez)):
        odgovori_b.append(fuzija(nisu_biljke_rez[s], van_teme_rez[t], K_RRF, W_SLIKA)["ishod"])
print(f"(b) van teme + nije biljka: {len(odgovori_b)} parova, odgovor fuzije (%):",
      (pd.Series(odgovori_b).value_counts(normalize=True) * 100).round(1).to_dict())

isto_slika = np.mean([fuzija(s, None)["prikaz"] == samo_jedna_grana(s)["prikaz"] for s in slika_rez]) * 100
isto_tekst = np.mean([fuzija(None, t)["prikaz"] == samo_jedna_grana(t)["prikaz"] for t in tekst_rez]) * 100
print(f"(c) samo slika: fuzija ista kao slika u {isto_slika:.1f}% | samo tekst: ista kao tekst u {isto_tekst:.1f}%")
print("    obrazloženje kad nema slike:", fuzija(None, tekst_rez[0])["obrazlozenje"])
print("    kad nema ničega:", fuzija(None, van_teme_rez[0])["obrazlozenje"])

assert isto == 100 and isto_slika == 100 and isto_tekst == 100

Tekstualna grana: 100%|██████████| 40/40 [00:03<00:00, 12.57it/s]


Odluke teksta za upite van teme: {'abstain': 40}

(a) van teme + biljka iz baze: 800 parova, fuzija ista kao sama slika u 100.0%
(b) van teme + nije biljka: 3200 parova, odgovor fuzije (%): {'nema_odgovora': 90.0, 'lista': 10.0}
(c) samo slika: fuzija ista kao slika u 100.0% | samo tekst: ista kao tekst u 100.0%
    obrazloženje kad nema slike: samo na osnovu opisa
    kad nema ničega: Neuspela identifikacija biljke. Pokušajte ponovo sa boljom slikom i/ili tekstom.
